# Session 5, Notebook 3: IRBC Exercise, Steady State and Loss Weighting

**Course:** Deep Learning for Solving Dynamic Models, University of Padova, 23-24 September 2026
**Session:** Day 1, 15:40-16:30, Scaling up: IRBC with DEQNs
**Slides:** `05_IRBC.pdf`
**Author:** Simon Scheidegger (HEC, University of Lausanne)

Three short tasks on the two-country model of [`05_01_IRBC_DEQN_smooth.ipynb`](05_01_IRBC_DEQN_smooth.ipynb) and [`05_02_IRBC_DEQN_irreversible.ipynb`](05_02_IRBC_DEQN_irreversible.ipynb). None of them trains a network for more than a minute.

1. **Task 1 (10 min).** Steady-state comparative statics in the lecture's calibration, the written counterpart of the finger exercise on the slides.
2. **Task 2 (10 min).** Inverse-loss weighting, from [Session 3](../03_nas_loss_balancing), applied to the residual magnitudes that `05_02` actually prints.
3. **Task 3 (10 min).** One re-weighted smoke run of `05_01`, and a sentence about what moved.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 13

## Task 1: steady-state comparative statics

The deterministic steady state of the IRBC model is pinned down by the Euler equation alone. With $z^j = 0$, $k_{t+1} = k_t$ (so all adjustment-cost terms vanish) and $\lambda_{t+1} = \lambda_t$,

$$
  1 = \beta\big(\zeta A\,k_{ss}^{\zeta-1} + 1 - \delta\big)
  \quad\Longrightarrow\quad
  k_{ss} = \left(\frac{1/\beta - 1 + \delta}{\zeta A}\right)^{1/(\zeta-1)},
  \qquad c_{ss} = A\,k_{ss}^{\zeta} - \delta\,k_{ss}.
$$

Neither the Pareto weights nor the $\gamma_j$ enter: at the steady state every country holds the same capital and consumes the same amount.

| Parameter | Symbol | Value |
|---|---|---|
| Discount factor | $\beta$ | 0.99 |
| Depreciation | $\delta$ | 0.01 |
| Capital share | $\zeta$ | 0.36 |
| Productivity level | $A$ | $(1/\beta - 1 + \delta)/\zeta \approx 0.0558$, so that $k_{ss} = 1$ |

The notebooks recompute $A$ from $\beta$, $\delta$, $\zeta$, so changing a parameter there leaves $k_{ss} = 1$ and moves $A$ instead. Both conventions are useful; this task asks for both.

In [2]:
def steady_state(beta, delta, zeta, A):
    k_ss = ((1.0 / beta - 1.0 + delta) / (zeta * A)) ** (1.0 / (zeta - 1.0))
    c_ss = A * k_ss ** zeta - delta * k_ss
    return k_ss, c_ss

def normalized_A(beta, delta, zeta):
    return (1.0 / beta - 1.0 + delta) / zeta

beta0, delta0, zeta0 = 0.99, 0.01, 0.36
A0 = normalized_A(beta0, delta0, zeta0)
k0, c0 = steady_state(beta0, delta0, zeta0, A0)
print(f"baseline: A = {A0:.6f}, k_ss = {k0:.4f}, c_ss = {c0:.6f}, I_ss/Y_ss = {delta0 * k0 / (A0 * k0 ** zeta0):.4f}")

baseline: A = 0.055836, k_ss = 1.0000, c_ss = 0.045836, I_ss/Y_ss = 0.1791


### Your turn

Double the depreciation rate to $\delta = 0.02$.

**(a)** Keep $A$ at its baseline value. Before computing, write down whether $k_{ss}$ rises or falls, then compute $k_{ss}$ and $c_{ss}$.

**(b)** Renormalize $A$ as the notebooks do. What is the new $A$? What happens to $k_{ss}$, to $c_{ss}$ and to the investment share $I_{ss}/Y_{ss} = \delta k_{ss}/(A k_{ss}^\zeta)$?

**(c)** Repeat (a) for $\beta = 0.95$ and for $\zeta = 0.30$, one at a time, and print one table with all four scenarios.

In [3]:
# (a) delta = 0.02, A fixed at A0
delta1 = 0.02
# k1, c1 = steady_state(...)

# (b) delta = 0.02, A renormalized
# A1 = normalized_A(...)

# (c) beta = 0.95 and zeta = 0.30 with A fixed at A0; one table


<hr style="border-top: 2px dashed #c0392b;">

### Stop. Attempt the task above before reading on.

<br>

<hr style="border-top: 2px dashed #c0392b;">

### Solution

In [4]:
delta1 = 0.02

# (a) A fixed
k1, c1 = steady_state(beta0, delta1, zeta0, A0)
print(f"(a) delta = {delta1}, A fixed:        k_ss = {k1:.4f}  ({100 * (k1 / k0 - 1):+.1f}%),  c_ss = {c1:.6f}  ({100 * (c1 / c0 - 1):+.1f}%)")

# (b) A renormalized
A1 = normalized_A(beta0, delta1, zeta0)
k1n, c1n = steady_state(beta0, delta1, zeta0, A1)
print(f"(b) delta = {delta1}, A renormalized: A = {A1:.6f}, k_ss = {k1n:.4f}, c_ss = {c1n:.6f}, "
      f"I_ss/Y_ss = {delta1 * k1n / (A1 * k1n ** zeta0):.4f}  (baseline {delta0 * k0 / (A0 * k0 ** zeta0):.4f})")

# (c) table, A fixed
scenarios = [
    ("baseline",        beta0, delta0, zeta0),
    ("delta = 0.02",    beta0, delta1, zeta0),
    ("beta = 0.95",     0.95,  delta0, zeta0),
    ("zeta = 0.30",     beta0, delta0, 0.30),
]
print(f"\n{'scenario':<14s} {'beta':>6s} {'delta':>6s} {'zeta':>5s} {'k_ss':>8s} {'c_ss':>9s} {'dk %':>7s} {'dc %':>7s}")
for name, b, d, z in scenarios:
    k, c = steady_state(b, d, z, A0)
    print(f"{name:<14s} {b:6.2f} {d:6.3f} {z:5.2f} {k:8.4f} {c:9.6f} {100 * (k / k0 - 1):+7.1f} {100 * (c / c0 - 1):+7.1f}")

(a) delta = 0.02, A fixed:        k_ss = 0.5321  (-46.8%),  c_ss = 0.033849  (-26.2%)
(b) delta = 0.02, A renormalized: A = 0.083614, k_ss = 1.0000, c_ss = 0.063614, I_ss/Y_ss = 0.2392  (baseline 0.1791)

scenario         beta  delta  zeta     k_ss      c_ss    dk %    dc %
baseline         0.99  0.010  0.36   1.0000  0.045836    +0.0    +0.0
delta = 0.02     0.99  0.020  0.36   0.5321  0.033849   -46.8   -26.2
beta = 0.95      0.95  0.010  0.36   0.1694  0.027770   -83.1   -39.4
zeta = 0.30      0.99  0.010  0.30   0.7707  0.043932   -22.9    -4.2


With $A$ fixed, a higher $\delta$ needs a higher marginal product of capital, so $k_{ss}$ falls, here by almost half; a lower $\beta$ works the same way through $1/\beta - 1$; a lower $\zeta$ lowers the marginal product at every $k$. With $A$ renormalized, $k_{ss}$ stays at one by construction and the change shows up in the investment share, which rises from 0.18 to 0.24 because twice as much capital has to be replaced each quarter.

## Task 2: inverse-loss weighting on the real residuals

The loss of `05_02` is $\mathcal L = w_E L_E + w_A L_A + w_F L_F$ with $L_E$, $L_A$, $L_F$ the mean squared Euler, resource and Fischer-Burmeister residuals, all with weight one. Session 3 introduced inverse-loss weighting, $w_i = (1/L_i)/\sum_j (1/L_j)$, which equalizes the contribution of every component.

The table below holds the mean absolute residuals that the stored teaching run of `05_02` prints at three points of training (segment 0, the end of the first learning-rate stage, and the last segment). Take the loss components to be the squares of these means.

| segment | mean $|$Euler$|$ | mean $|$ARC$|$ | mean $|$FB$|$ |
|---|---|---|---|
| 0 | 2.84e-03 | 4.72e-04 | 4.47e-05 |
| 160 | 2.05e-05 | 4.50e-05 | 4.25e-05 |
| 400 | 1.90e-05 | 4.57e-05 | 4.23e-05 |

**(a)** For each of the three rows, compute the inverse-loss weights and the share of the equal-weight loss that each component accounts for.

**(b)** Which component receives the largest inverse-loss weight, and does that make sense here? Recall from `05_02` that wherever the constraint is slack the Fischer-Burmeister residual equals $\mu/\lambda$ to first order, and $\mu$ sits at the floor of its softplus head ($4.5\times10^{-5}$), so every FB value in the table is that floor.

In [5]:
log_rows = {
    0: dict(Euler=2.840e-03, ARC=4.720e-04, FB=4.470e-05),
    160: dict(Euler=2.050e-05, ARC=4.500e-05, FB=4.250e-05),
    400: dict(Euler=1.900e-05, ARC=4.570e-05, FB=4.230e-05),
}

def inverse_loss_weights(losses):
    # losses: dict name -> mean squared residual
    # return: dict name -> weight, normalized to sum to one
    raise NotImplementedError

# for seg, row in log_rows.items():
#     losses = {name: val ** 2 for name, val in row.items()}
#     ...


<hr style="border-top: 2px dashed #c0392b;">

### Stop. Attempt the task above before reading on.

<br>

<hr style="border-top: 2px dashed #c0392b;">

### Solution

In [6]:
def inverse_loss_weights(losses):
    inv = {name: 1.0 / val for name, val in losses.items()}
    total = sum(inv.values())
    return {name: v / total for name, v in inv.items()}

for seg, row in log_rows.items():
    losses = {name: val ** 2 for name, val in row.items()}
    w = inverse_loss_weights(losses)
    total = sum(losses.values())
    print(f"segment {seg}")
    print(f"  {'component':<8s} {'L_i':>10s} {'share of equal-weight loss':>28s} {'inverse-loss weight':>20s}")
    for name in losses:
        print(f"  {name:<8s} {losses[name]:10.2e} {100 * losses[name] / total:27.1f}% {w[name]:20.4f}")

segment 0
  component        L_i   share of equal-weight loss  inverse-loss weight
  Euler      8.07e-06                        97.3%               0.0002
  ARC        2.23e-07                         2.7%               0.0089
  FB         2.00e-09                         0.0%               0.9909
segment 160
  component        L_i   share of equal-weight loss  inverse-loss weight
  Euler      4.20e-10                         9.9%               0.6944
  ARC        2.03e-09                        47.6%               0.1441
  FB         1.81e-09                        42.5%               0.1616
segment 400
  component        L_i   share of equal-weight loss  inverse-loss weight
  Euler      3.61e-10                         8.5%               0.7275
  ARC        2.09e-09                        49.3%               0.1257
  FB         1.79e-09                        42.2%               0.1468


At segment 0 the Euler component is 97 percent of the loss, and inverse-loss weighting hands 99 percent of the weight to the Fischer-Burmeister term. By segment 160 the Euler residual has fallen by two orders of magnitude and is the smallest component, so it now receives the largest weight, about 0.7, with the resource and Fischer-Burmeister terms sharing the rest.

Two things to notice. The weight that goes to the Fischer-Burmeister term, 99 percent at the start and 15 percent at the end, is wasted: its value is the floor of the multiplier head, not a sign that complementarity is neglected, and no weight can lower it. And at the end the rule puts the most weight on the component that has already converged furthest, which is the rule working as designed, not obviously what one wants. Inverse-loss weighting is a good default when every component can still fall; a component pinned at a floor has to be excluded from the rule or measured differently, here by the product $(\mu/\lambda)(I/k)$, which does go to zero.

## Task 3: one re-weighted run

Open `05_01_IRBC_DEQN_smooth.ipynb`, set `RUN_MODE = "smoke"`, and run it once as is and once with `ARC_WEIGHT = 10.0` in the configuration cell. Each run takes under a minute. Compare the two residual panels (mean $|$Euler$|$ and mean $|$ARC$|$ against the training segment) and the residual report on the ergodic set.

Write one sentence: which residual moved, in which direction, and what happened to the other one?

<hr style="border-top: 2px dashed #c0392b;">

### Stop. Attempt the task above before reading on.

<br>

<hr style="border-top: 2px dashed #c0392b;">

### Solution

Two smoke runs on the same seed give, on the ergodic set:

| `ARC_WEIGHT` | mean $|$Euler$|$ | mean $|$ARC$|$ |
|---|---|---|
| 1 | 6.58e-04 | 5.77e-05 |
| 10 | 6.84e-04 | 4.24e-05 |

Raising the weight on the resource residual tenfold lowers it by about a quarter and leaves the Euler residual where it was. At equal weights the resource term was already the smaller component, so extra weight on it buys little; weighting matters for a component that is large and stubborn, not for one that is small already.